# Data Voorbereiding

Notebook voor het inlezen, samenvoegen en valideren van alle brondata.

| Bron | Script | Beschrijving |
|---|---|---|
| Solarlogs | `solar_logs` | Uurlijkse afname/injectie/productie per dag (iLumen API) |
| SolarBattery | `battery` | Uurlijkse geladen/ontladen/SOC per dag (iLumen API) |
| Fluvius | `fluvius` | Kwartiertotalen afname/injectie dag+nacht (digitale meter) |
| OwnDev | `owndev` | Seconde-tijdreeks P1 + SOFAR + commando's (Raspberry Pi) |
| Solarcharge | `solarcharge` | EV laadsessies uitgespreid per kwartier (iLuCharge) |
| Weer | `weather` | Uurlijkse Open-Meteo data + pvlib POA instraling |

In [ ]:
# ── Pad instellen ─────────────────────────────────────────────────────────
# De notebook ligt in notebooks/, de scripts in scripts/ één niveau hoger.
# sys.path.insert zorgt dat Python de scripts-map kan vinden zonder installatie.
import sys
sys.path.insert(0, "..")  # scripts/ is één niveau omhoog

# ── Standaardbibliotheken ──────────────────────────────────────────────────
import pandas as pd          # dataframes, CSV lezen/schrijven, tijdreeksen
import matplotlib.pyplot as plt  # grafieken en visualisaties

# ── Eigen scripts per databron ─────────────────────────────────────────────
# Elke module bevat de laad- en verwerkingsfuncties voor één databron.
from scripts import solar_logs   # iLumen API – uurlijkse zon- en netstroom
from scripts import battery      # iLumen API – uurlijkse batterijdata
from scripts import fluvius      # Fluvius CSV – kwartiertotalen digitale meter
from scripts import owndev       # OwnDev Raspberry Pi – seconde-telegrammen P1 + SOFAR
from scripts import solarcharge  # iLuCharge API – EV laadsessies per kwartier
from scripts import weather      # Open-Meteo + pvlib – uurlijks weer en instraling

# ── Configuratiepaden ──────────────────────────────────────────────────────
# Gecentraliseerd in scripts/config.py zodat alle scripts dezelfde mappen gebruiken.
from scripts.config import SOLAR_DIR, BATTERY_DIR, INTERMEDIATE_DIR, WEATHER_CSV

## 1. Beschikbare data per bron

In [ ]:
# ── Beschikbare datums ophalen per bron ───────────────────────────────────
# Elke functie scant de lokale map en geeft een gesorteerde lijst van datums terug.
# Dit geeft een snel overzicht van hoeveel historische data er beschikbaar is.
solar_dates   = solar_logs.available_dates()   # lijst van datetime.date objecten
battery_dates = battery.available_dates()      # idem voor batterijdata

# ── Volledige dataframes laden ─────────────────────────────────────────────
# Fluvius en OwnDev bevatten alle data in één CSV; direct inladen als DataFrame.
df_fl = fluvius.laad()   # kolommen: kwartier, afname_dag, afname_nacht, terugave_dag, terugave_nacht
df_od = owndev.laad()    # kolommen: tijdstip, afname_kw, terugave_kw, bat_laden_kw, bat_ontladen_kw, soc, sofar_action, commando_kw

# ── EV laadsessies en weerdata ─────────────────────────────────────────────
df_ev   = solarcharge.available_sessions()  # kwartierrijen EV-lading; None als geen data beschikbaar
weer_ok = WEATHER_CSV.exists()              # simpele check – weerfunctie heeft geen afzonderlijke laadmethode

# ── Overzicht afdrukken ────────────────────────────────────────────────────
# Toont per bron het aantal rijen/dagen en de gedekte periode.
# Zo is in één oogopslag te zien of alle bronnen up-to-date en compleet zijn.
print(f"Solarlogs    : {len(solar_dates)} dagen   ({solar_dates[0]} → {solar_dates[-1]})")
print(f"Batterij     : {len(battery_dates)} dagen   ({battery_dates[0]} → {battery_dates[-1]})")
print(f"Fluvius      : {len(df_fl):,} kwartieren ({df_fl['kwartier'].min().date()} → {df_fl['kwartier'].max().date()})")
print(f"OwnDev       : {len(df_od):,} seconden   ({df_od['tijdstip'].min()} → {df_od['tijdstip'].max()})")
print(f"EV sessies   : {len(df_ev) if df_ev is not None else 0} kwartierrijen")
print(f"Weerdata     : {'aanwezig' if weer_ok else 'ONTBREEKT'}")

## 2. OwnDev — telegrammen verwerken naar seconde-CSV

In [ ]:
# ── Telegrammen verwerken naar seconde-CSV ────────────────────────────────
# owndev.verwerk() doorloopt alle telegram-bestanden in data/Source Data/OwnDev/
# en schrijft de verwerkte rijen incrementeel weg naar owndev_seconden.csv.
#
# Incrementeel: bestanden waarvan de bestandsnaam-tijdstempel ≤ de laatste
# al verwerkte tijdstempel zijn worden overgeslagen. Zo is heruitvoeren veilig.
#
# Geeft terug:
#   pad    – Path naar het outputbestand
#   n_nieuw – aantal nieuw toegevoegde rijen (0 als alles al verwerkt was)
pad, n_nieuw = owndev.verwerk()

print(f"Outputbestand : {pad}")
print(f"Nieuwe rijen  : {n_nieuw:,}")

# ── Verwerkt bestand opnieuw inladen ──────────────────────────────────────
# df_od wordt herladen zodat het de meest recente versie van de CSV bevat,
# ook als owndev.verwerk() net nieuwe rijen heeft toegevoegd.
df_od = owndev.laad()
print(f"Totaal rijen  : {len(df_od):,}")
print(f"Periode       : {df_od['tijdstip'].min()} → {df_od['tijdstip'].max()}")
print(f"Kolommen      : {list(df_od.columns)}")

# Eerste rijen tonen als snelle sanity-check op inhoud en kolomnamen
df_od.head()

## 3. OwnDev — nuttige commando's en respons per seconde

In [ ]:
# ── Module herladen ────────────────────────────────────────────────────────
# importlib.reload() is nodig wanneer scripts/owndev.py aangepast werd
# nadat de notebook al gestart was. Zonder reload blijft de oude versie
# in het geheugen en zijn nieuwe functies niet zichtbaar.
#
# Volgorde is belangrijk:
#   1. reload() – laad de gewijzigde broncode opnieuw in
#   2. from scripts import owndev – herlaad de module-referentie
#   3. from scripts.owndev import ... – haal de constanten op uit de herlaadde versie
import importlib
import scripts.owndev
importlib.reload(scripts.owndev)

from scripts import owndev

# RESPONS_FILE – pad naar commando_respons.csv (tussenresultaat)
# OUTPUT_FILE  – pad naar owndev_seconden.csv (basisdata per seconde)
from scripts.owndev import RESPONS_FILE, OUTPUT_FILE

In [ ]:
# ── Commando-respons analyseren ────────────────────────────────────────────
# analyseer_commando_respons() zoekt in owndev_seconden.csv naar "nuttige
# commando's": momenten waarop commando_kw verandert ten opzichte van de
# vorige waarde. Voor elk zo'n moment worden de 5 seconden erna opgeslagen
# als kolommen net_kw_s1..s5, bat_kw_s1..s5 en afwijking_kw_s1..s5.
#
# Kolomconventie (teken):
#   net_kw  = afname_kw − terugave_kw  → positief = afname van het net
#   bat_kw  = bat_laden_kw − bat_ontladen_kw → positief = opladen, negatief = ontladen
#   afwijking_kw = bat_kw − commando_kw → 0 als batterij exact doet wat gevraagd
#
# soc = State of Charge (%) gemeten op het moment van het commando zelf
#
# Geeft terug:
#   df_respons – DataFrame met één rij per nuttig commando
#   pad        – Path naar het opgeslagen CSV-bestand
df_respons, pad = owndev.analyseer_commando_respons()

# ── Overzicht afdrukken ────────────────────────────────────────────────────
print(f"Outputbestand   : {pad}")
print(f"Nuttige commando's: {len(df_respons)}")      # totaal aantal commando-wisselingen
print(f"Kolommen          : {list(df_respons.columns)}")
print()

# Verdeling per actie-type geeft inzicht in hoe de batterij gestuurd werd
print("Verdeling per actie:")
print(df_respons["sofar_action"].value_counts().to_string())
print()

# Eerste 10 rijen tonen – inclusief NaN-rijen aan het begin (geen commando bekend)
df_respons.head(10)

In [ ]:
# ── Resultaat opslaan naar CSV ─────────────────────────────────────────────
# Sla df_respons expliciet op zodat andere notebooks of scripts de commando-
# respons kunnen inladen zonder de volledige analyse opnieuw te draaien.
# date_format zorgt voor een leesbaar en parseerbaar tijdstempelformaat.
df_respons.to_csv(RESPONS_FILE, index=False, date_format="%Y-%m-%d %H:%M:%S")

# Bevestiging: pad, aantal rijen en bestandsgrootte ter controle
print(f"Opgeslagen: {RESPONS_FILE}")
print(f"Rijen     : {len(df_respons):,}")
print(f"Grootte   : {RESPONS_FILE.stat().st_size / 1024:.1f} KB")

# Eerste rijen tonen als visuele check
df_respons.head()

## 4. Gemiddelde en maximale afwijking per seconde

De grafiek toont voor elk SOFAR-commando-type hoe nauwkeurig de batterij het gevraagde vermogen opvolgt, gemeten over de 5 seconden na het commando.

**Wat wordt gemeten?**
De *afwijking* is het verschil tussen het effectief geleverde batterijvermogen (`bat_kw`) en het gevraagde vermogen (`commando_kw`). Een afwijking van 0 betekent dat de batterij precies doet wat gevraagd werd.

**Links — gemiddelde afwijking (kW)**
Geeft aan of de batterij systematisch te veel of te weinig levert. Waarden dicht bij 0 zijn goed. Een negatieve waarde bij een laadcommando betekent dat de batterij minder laadt dan gevraagd; een positieve waarde bij ontladen betekent dat ze minder ontlaadt dan gevraagd.

**Rechts — maximale absolute afwijking (kW)**
Toont de grootste uitwijking die voorkwam. Dit is een maat voor de worst-case nauwkeurigheid. Hoge waarden wijzen op incidentele fouten of trage respons bij bepaalde commando-types.

**Kleur per commando-type**
Elk commando-type (`laden tot voorziene level`, `ontladen tot voorziene level`, `laden door zon`, …) heeft een eigen kleur zodat de prestaties per type vergelijkbaar zijn.

> **Verwachting:** de afwijking zou per seconde kleiner moeten worden naarmate de batterij tijd heeft om op het commando te reageren. Als de waarden na seconde 1 al stabiel zijn, reageert het systeem snel.

In [ ]:
# ── Module opnieuw laden voor het geval owndev.py gewijzigd werd ──────────
importlib.reload(scripts.owndev)
from scripts import owndev
from scripts.owndev import RESPONS_FILE

# ── Afwijking per commando berekenen ──────────────────────────────────────
# afwijking_per_commando() leest commando_respons.csv en groepeert per
# combinatie van sofar_action × seconde (s1 t/m s5).
#
# Per groep worden berekend:
#   gem_afwijking – gemiddelde van afwijking_kw over alle commando-momenten
#                   → negatief = batterij levert systematisch minder dan gevraagd
#   max_afwijking – maximale absolute afwijking (worst-case fout)
#   n             – aantal meetpunten in die groep
#
# Het resultaat is een "lang" DataFrame: 4 acties × 5 seconden = 20 rijen.
df_afw = owndev.afwijking_per_commando()
df_afw

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Unieke acties en seconden uit het resultaat-DataFrame ─────────────────
acties   = df_afw["sofar_action"].unique()   # bijv. ['laden door zon', 'stoppen', ...]
seconden = sorted(df_afw["seconde"].unique()) # [1, 2, 3, 4, 5]
n_acties = len(acties)

# ── Kleurpalet: één kleur per actie-type ──────────────────────────────────
# tab10 heeft 10 onderscheidbare kleuren; linspace(0, 0.6) haalt de eerste
# n_acties kleuren op zonder de donkere tinten helemaal rechts te gebruiken.
kleuren = plt.cm.tab10(np.linspace(0, 0.6, n_acties))

# ── Figuur aanmaken: twee naast elkaar ────────────────────────────────────
# Links: gemiddelde afwijking | Rechts: maximale absolute afwijking
# sharey=False want de schalen van gem en max zijn heel verschillend
fig, assen = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
fig.suptitle("Batterij-afwijking per commando-type (geleverd − gevraagd vermogen)", fontsize=13)

# ── Breedte en positie van de staafjes berekenen ──────────────────────────
# 0.8 van een eenheid verdeeld over n_acties zodat alle staafjes naast
# elkaar passen zonder overlap. x is de middenpositie per seconde-groep.
breedte = 0.8 / n_acties
x       = np.arange(len(seconden))   # [0, 1, 2, 3, 4] → x-positie per seconde

# ── Staafjes tekenen per actie ─────────────────────────────────────────────
for i, actie in enumerate(acties):
    # Selecteer rijen voor deze actie en indexeer op seconde voor snelle lookup
    sub = df_afw[df_afw["sofar_action"] == actie].set_index("seconde")

    # Haal gem en max per seconde op; 0 als een seconde ontbreekt (veiligheidsval)
    gem = [sub.loc[s, "gem_afwijking"] if s in sub.index else 0 for s in seconden]
    mx  = [sub.loc[s, "max_afwijking"] if s in sub.index else 0 for s in seconden]

    # Bereken de horizontale verschuiving zodat staafjes naast elkaar vallen
    # i=0 staat links, i=n_acties-1 staat rechts van het midden
    offset = (i - n_acties / 2 + 0.5) * breedte

    assen[0].bar(x + offset, gem, width=breedte, label=actie, color=kleuren[i])
    assen[1].bar(x + offset, mx,  width=breedte, label=actie, color=kleuren[i])

# ── Assen opmaken ─────────────────────────────────────────────────────────
# zip() koppelt elke as aan zijn titel en eenheid in één loop
for ax, titel, eenheid in zip(
    assen,
    ["Gemiddelde afwijking (kW)", "Maximale absolute afwijking (kW)"],
    ["kW", "kW"],
):
    ax.set_title(titel)
    ax.set_xlabel("Seconde na commando")
    ax.set_ylabel(eenheid)
    ax.set_xticks(x)
    ax.set_xticklabels([f"s{s}" for s in seconden])  # "s1", "s2", ... als label
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")  # nul-lijn als referentie
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)   # horizontale hulplijnen voor leesbaarheid

plt.tight_layout()
plt.show()

## 5. Outliers in de afwijking

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ══════════════════════════════════════════════════════════════════════════
# PARAMETERS
# ══════════════════════════════════════════════════════════════════════════
# DREMPEL_FACTOR bepaalt hoe ver een punt van het gemiddelde moet afwijken
# om als outlier beschouwd te worden.
# Formule: |afwijking − gemiddelde| > DREMPEL_FACTOR × standaardafwijking
# Waarde 2 = ~5% kans bij een normaalverdeling → streng maar gangbaar
DREMPEL_FACTOR = 2

# Lijst van de 5 afwijkingskolommen (één per seconde na het commando)
afw_kolommen = [f"afwijking_kw_s{n}" for n in range(1, 6)]

# Gesorteerde lijst van alle acties voor een vaste kleur-actie-koppeling
acties_gesorteerd = sorted(df_respons["sofar_action"].dropna().unique())

# Bouw een woordenboek {actie: kleur} zodat elke actie in alle deelvensters
# dezelfde kleur heeft en de legende consistent is
kleur_map = dict(zip(
    acties_gesorteerd,
    plt.cm.tab10(np.linspace(0, 0.6, len(acties_gesorteerd)))
))

# ══════════════════════════════════════════════════════════════════════════
# OUTLIER-STATISTIEKEN BEREKENEN
# ══════════════════════════════════════════════════════════════════════════
# Voor elke combinatie van seconde × actie:
#   1. Haal alle afwijkingswaarden op (NaN verwijderd)
#   2. Bereken gemiddelde en standaardafwijking
#   3. Markeer punten buiten gem ± DREMPEL_FACTOR×std als outlier
#   4. Sla statistieken + ruwe waarden op voor plot én tabel
rapport = []
for kol in afw_kolommen:
    seconde = int(kol.split("_s")[1])          # "afwijking_kw_s3" → 3
    data    = df_respons[["sofar_action", kol]].dropna()

    for actie, groep in data.groupby("sofar_action"):
        waarden = groep[kol].values
        gem     = waarden.mean()
        std     = waarden.std()

        # Boolean masker: True voor outliers
        is_out = np.abs(waarden - gem) > DREMPEL_FACTOR * std
        n_out  = int(is_out.sum())

        rapport.append({
            "seconde":      seconde,
            "sofar_action": actie,
            "n_totaal":     len(waarden),
            "n_outliers":   n_out,
            "pct_outliers": round(100 * n_out / len(waarden), 1),
            "gem":          round(gem, 4),
            "std":          round(std, 4),
            "drempel":      round(DREMPEL_FACTOR * std, 4),
            # Ruwe arrays bewaard voor de strip-plot hieronder
            "_waarden":     waarden,
            "_is_out":      is_out,
        })

df_rapport = pd.DataFrame(rapport)

# ══════════════════════════════════════════════════════════════════════════
# STRIP-PLOT: één deelvenster per seconde
# ══════════════════════════════════════════════════════════════════════════
# Elke kolom = één seconde. Binnen een kolom staan de 4 actie-types naast
# elkaar als verticale puntenwolken. sharey=True zodat de y-schaal gelijk
# blijft over alle seconden en vergelijking mogelijk is.
fig, assen = plt.subplots(1, len(afw_kolommen), figsize=(15, 5), sharey=True)
fig.suptitle(
    f"Outliers per seconde na commando  (drempel = gem ± {DREMPEL_FACTOR}×std)",
    fontsize=12,
)

for ax, kol in zip(assen, afw_kolommen):
    seconde = int(kol.split("_s")[1])
    rijen   = df_rapport[df_rapport["seconde"] == seconde]
    x_pos   = 0   # elke actie krijgt een eigen x-positie binnen het deelvenster

    for _, r in rijen.iterrows():
        kleur   = kleur_map[r["sofar_action"]]
        normaal = r["_waarden"][~r["_is_out"]]   # gewone punten
        uitbijt = r["_waarden"][ r["_is_out"]]   # outliers

        # Normale metingen: kleine, transparante punten
        # Door de lage alpha smelten overlappende punten samen tot een "wolk";
        # hoe dichter de punten op elkaar liggen, hoe donkerder de wolk wordt.
        # De visuele dikte van de wolk weerspiegelt dus de dichtheid van de verdeling.
        ax.scatter([x_pos] * len(normaal), normaal,
                   s=4, alpha=0.15, color=kleur)

        # Outliers: grotere diamanten, goed zichtbaar bovenop de wolk (zorder=5)
        # In de legenda: naam van de actie + (aantal, percentage) voor snelle lezing
        if len(uitbijt):
            ax.scatter([x_pos] * len(uitbijt), uitbijt,
                       s=25, alpha=0.85, color=kleur, marker="D", zorder=5,
                       label=f"{r['sofar_action']} ({r['n_outliers']}, {r['pct_outliers']}%)")

        # Horizontale lijn op het gemiddelde van deze actie in dit deelvenster.
        # De lijndikte (linewidth=1.5) is voor alle acties gelijk; de positie
        # op de y-as laat zien of het gemiddelde boven of onder nul ligt.
        ax.hlines(r["gem"], x_pos - 0.3, x_pos + 0.3,
                  colors=kleur, linewidth=1.5)

        x_pos += 1   # volgende actie naast de vorige

    # Stippellijn op y=0: dit is het ideaal (batterij doet exact wat gevraagd)
    ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
    ax.set_title(f"+{seconde}s", fontsize=10)
    ax.set_xticks([])   # x-as heeft geen labels – actie-namen staan in de legenda
    ax.grid(axis="y", alpha=0.3)

    # Legenda alleen tonen als er outliers zijn (anders is ze leeg)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=6, loc="upper right",
                  title="actie (n, %)", title_fontsize=6)

assen[0].set_ylabel("Afwijking (kW)")
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# OVERZICHTSTABEL
# ══════════════════════════════════════════════════════════════════════════
# Sorteren op actie + seconde voor een leesbare tabel.
# n_outliers en pct_outliers geven snel inzicht in hoe vaak extreme
# afwijkingen voorkomen bij elk commando-type en elke seconde.
print(f"\nOutlier-rapport (drempel = {DREMPEL_FACTOR}×std):\n")
print(
    df_rapport[["sofar_action", "seconde", "n_totaal", "n_outliers",
                "pct_outliers", "gem", "std", "drempel"]]
    .sort_values(["sofar_action", "seconde"])
    .to_string(index=False)
)

### Uitleg van de grafiek

**Structuur**
De grafiek bestaat uit 5 deelvensters, één per seconde na het commando (+1s t/m +5s). Binnen elk venster staan de 4 commando-types naast elkaar op de x-as. De y-as toont de afwijking in kW (positief = batterij levert meer dan gevraagd, negatief = batterij levert minder).

---

**Betekenis van de visuele elementen**

| Element | Wat het is | Wat het betekent |
|---|---|---|
| **Kleine ronde punten** | Alle individuele metingen (normaal) | Elke punt is één commando-moment. De kleur geeft het commando-type aan. |
| **Dikte van de puntenwolk** | Concentratie van waarden | Hoe dikker de wolk, hoe meer metingen op die waarde clusteren. Een smalle wolk duidt op een consistente, voorspelbare respons; een brede wolk wijst op grote variatie. |
| **Diamanten (◆)** | Outliers | Punten waarvoor \|afwijking − gemiddelde\| > 2 × standaardafwijking. Ze liggen duidelijk buiten de normale spreiding. Tussen haakjes staat het aantal en het percentage. |
| **Horizontale streep (—)** | Gemiddelde afwijking | De dikte van de streep (1.5 pt) is voor alle acties gelijk; alleen de positie op de y-as verschilt. Een streep precies op 0 betekent dat de batterij gemiddeld exact het gevraagde vermogen levert. |

---

**Analyse per commando-type**

**`stoppen`**
De puntenwolk zit uiterst smal gebundeld net boven 0 (≈ +0.05 kW). Dat kleine positieve gemiddelde is een restant van de minimale batterijstroom die de SOFAR-omvormer altijd trekt bij "stoppen". Er zijn vrijwel geen outliers. De batterij reageert op dit commando vrijwel onmiddellijk en met hoge herhaalnauwkeurigheid — het gedrag is over alle 5 seconden identiek.

**`laden door zon`**
De wolk is iets breder dan bij stoppen, maar blijft geconcentreerd rond een kleine negatieve afwijking (≈ −0.02 kW). Dit commando wordt gestuurd op basis van de actuele PV-productie en varieert dus in grootte; toch is de spreiding gering. Enkele schaarse outliers zijn zichtbaar, maar hun percentage blijft laag. Het systeem volgt dit commando goed op over alle 5 seconden.

**`ontladen tot voorziene level`**
De wolk is merkbaar breder: de batterij ontlaadt niet altijd exact het gevraagde vermogen, maar wijkt gemiddeld slechts licht af (≈ −0.03 kW). De outliers zijn iets talrijker dan bij laden door zon, maar de gemiddelde streep blijft stabiel over de 5 seconden — er is geen toenemende afwijking naarmate de tijd vordert. De spreiding is vermoedelijk deels te verklaren door variatie in het netverbruik op het moment van het commando.

**`laden tot voorziene level`**
Dit type vertoont de breedste puntenwolk én de meeste uitschieters. Het gemiddelde is stelselmatig negatief (≈ −0.10 kW) en de maximale outliers groeien toe naar s5 (tot > 5 kW). De verklaring is dat dit commando wordt afgegeven wanneer de batterij een vooraf bepaald laadvermogen moet aanhouden; op het moment dat het doelvermogen bereikt is, schakelt de regelaar terug en ontstaat een korte piek in de afwijking. De outliers bij s4–s5 zijn dan ook geen aanloopfout maar een afsluiteffect: de regelaar is aan het terugschakelen terwijl de meting nog de overgangswaarde registreert.

---

**Besluit**

De batterij volgt de meeste commando's binnen enkele honderdsten kW nauwkeurig op. De kleine maar consistente negatieve afwijking over alle commando-types en seconden wijst op een vaste kalibratieafwijking in de SOFAR-regelaar — niet op een trage of verslechterende respons in de tijd.

De grootste onnauwkeurigheid doet zich voor bij `laden tot voorziene level`, specifiek aan het einde van de responstijd (s4–s5). Dit is inherent aan het regelgedrag bij het bereiken van het doelvermogen en is geen teken van een defect.

> **Aanbeveling:** de systematische offset (0.05–0.10 kW) is verwaarloosbaar voor analyses op dagtotaalniveau. Voor vermogenssturing op secondeniveau — bijvoorbeeld om pieken te vermijden — is een correctiefactor in de commando-berekening zinvol.